# The rat and cat gene spaces


**What this builds (instruction 53, WP4-6).** One row per gene for four species, from their Ensembl builds: id, symbol, biotype, chromosome, start, end, strand. Rat and cat are Toxoplasma hosts; Anopheles gambiae is the VECTOR that carries Plasmodium through its sexual cycle; P. vivax is the other human malaria parasite, kept apart from falciparum because it forms hypnozoites and falciparum does not.

**Both were recorded as blocked behind the VEuPathDB login, and that was wrong.** A rat and a cat are vertebrates; their builds are on Ensembl and nothing about them needs a parasite database. The assumption went unchecked for weeks.

**A gene space is not an answer.** No rat or cat measurement is on disk, so both spaces are declared and installable, and neither is calibrated. The cat matters out of proportion to its data: it is the only species where Toxoplasma completes its sexual cycle, so the enteroepithelial stages exist nowhere else.

Built 2026-11-09.

In [1]:
import gzip, hashlib, json, os, re, pandas as pd
from parascape import paths
from parascape.spaces.anopheles import ANOPHELES_SPACE
from parascape.spaces.cat import CAT_SPACE
from parascape.spaces.rat import RAT_SPACE
from parascape.spaces.stephensi import STEPHENSI_SPACE
from parascape.spaces.vivax import VIVAX_SPACE
root = paths.dataset_root()
SOURCES = {RAT_SPACE: ('reference/ensembl_r113', 'Rattus_norvegicus.mRatBN7.2.113.gff3.gz'),
           CAT_SPACE: ('reference/ensembl_r113', 'Felis_catus.Felis_catus_9.0.113.gff3.gz'),
           ANOPHELES_SPACE: ('reference/ensembl_metazoa_r60', 'Anopheles_gambiae.AgamP4.60.chr.gff3.gz'),
           STEPHENSI_SPACE: ('reference/ensembl_metazoa_r60', 'Anopheles_stephensi.AsteS1.60.gff3.gz'),
           VIVAX_SPACE: ('reference/ensembl_protists_r63',
                         'Plasmodium_vivax_gca900093555.GCA900093555v2.63.gff3.gz')}
def sha(path):
    return hashlib.sha256(open(path, 'rb').read()).hexdigest()
paths_by_space = {s: os.path.join(root, where, f) for s, (where, f) in SOURCES.items()}
for space, path in paths_by_space.items():
    print(space.code, os.path.basename(path), 'present:', os.path.exists(path))
hashes = {s.code: sha(p) for s, p in paths_by_space.items() if os.path.exists(p)}
pd.Series(hashes).to_frame('sha256 of the build')

Rn Rattus_norvegicus.mRatBN7.2.113.gff3.gz present: True
Fc Felis_catus.Felis_catus_9.0.113.gff3.gz present: True
Ag Anopheles_gambiae.AgamP4.60.chr.gff3.gz present: True
As Anopheles_stephensi.AsteS1.60.gff3.gz present: True
Pv Plasmodium_vivax_gca900093555.GCA900093555v2.63.gff3.gz present: True


,sha256 of the build
Rn,3f7909df473a03e65af12c37bf7bb7e42a89623d14053224e78fb82168eae3c4
Fc,aa4a27e48a358078e23b4a2c37b33105e4366aa5e2495364b6ea6bcde7968068
Ag,f6c881106bccffb87406836a9e579811de2ad51809d6e4b5785fdd9feef212bf
As,b9919cf879f18a9962f929d9381f97c5008fbebe7b5a14bf8daca18bcf1430bc
Pv,f1dc232c5911159466a1cf895f4d7a888d137ffe1c1f0c086e633a7e812d03c8


## 1 · One row per gene

Genes only, not transcripts: a gene space is keyed by gene. The id is the Ensembl stable id with no version, which is what the space's pattern matches.

In [2]:
def genes_of(path):
    rows = []
    with gzip.open(path, 'rt') as handle:
        for line in handle:
            if line.startswith('#'):
                continue
            field = line.rstrip('\n').split('\t')
            if len(field) < 9 or field[2] != 'gene':
                continue
            attributes = field[8]
            found = re.search(r'ID=gene:([^;]+)', attributes)
            name = re.search(r'Name=([^;]+)', attributes)
            biotype = re.search(r'biotype=([^;]+)', attributes)
            rows.append(dict(gene_id=(found.group(1) if found else ''),
                             gene_name=(name.group(1) if name else ''),
                             biotype=(biotype.group(1) if biotype else ''),
                             seqid=field[0], start=int(field[3]), end=int(field[4]),
                             strand=field[6]))
    frame = pd.DataFrame(rows)
    return frame[frame.gene_id != ''].drop_duplicates('gene_id').reset_index(drop=True)
tables = {space: genes_of(path) for space, path in paths_by_space.items() if os.path.exists(path)}
summary = pd.DataFrame([dict(space=s.code, species=s.species, genes=len(t),
                             protein_coding=int((t.biotype == 'protein_coding').sum()),
                             named=int((t.gene_name != '').sum()),
                             ids_canonical=bool(t.gene_id.map(s.matches).all()))
                        for s, t in tables.items()])
summary

,space,species,genes,protein_coding,named,ids_canonical
0,Rn,Rattus norvegicus,23141,23098,21346,True
1,Fc,Felis catus,19588,19564,16010,True
2,Ag,Anopheles gambiae,12614,12614,1839,True
3,As,Anopheles stephensi,13336,13336,579,True
4,Pv,Plasmodium vivax,6523,6523,1907,True


## 2 · What Bgee says each gene is expressed in

The same two columns the mouse space carries, so the three host spaces answer the same question the same way: the best expression score over the tissues a gene is called present in, and how many tissues those are. A gene called absent everywhere keeps a score and a count of zero -- Bgee tested it, so that is a measurement, not a gap. A gene Bgee never tested stays missing.

In [3]:
BGEE = os.path.join('host', 'bgee_expression')
bgee_files = {RAT_SPACE: 'Rattus_norvegicus_expr_simple.tsv.gz',
              CAT_SPACE: 'Felis_catus_expr_simple.tsv.gz'}
for space, table in list(tables.items()):
    if space not in bgee_files:
        continue                       # Bgee covers vertebrates; the mosquito and the parasite are not in it
    path = os.path.join(root, BGEE, bgee_files[space])
    if not os.path.exists(path):
        print(space.code, 'no Bgee file'); continue
    calls = pd.read_csv(path, sep='	')
    calls.columns = [c.strip('"') for c in calls.columns]
    calls['score'] = pd.to_numeric(calls['Expression score'], errors='coerce')
    present = calls[calls.Expression == 'present']
    top = present.groupby('Gene ID').score.max()
    count = present.groupby('Gene ID').size()
    tested = calls['Gene ID'].drop_duplicates()
    table['bgee_top_score'] = table.gene_id.map(top)
    table['bgee_tissues_present'] = table.gene_id.map(count)
    # Tested and never present is a measured zero; never tested stays missing.
    was_tested = table.gene_id.isin(set(tested))
    for column in ('bgee_top_score', 'bgee_tissues_present'):
        table.loc[was_tested & table[column].isna(), column] = 0
    hashes[space.code + '_bgee'] = sha(path)
    print(f'{space.code}: {int(was_tested.sum())} of {len(table)} genes tested by Bgee, '
          f'{int(table.bgee_tissues_present.gt(0).sum())} expressed somewhere, '
          f'{calls['Anatomical entity ID'].nunique()} tissues')
pd.DataFrame({s.code: t[['bgee_top_score', 'bgee_tissues_present']].notna().sum()
              for s, t in tables.items() if 'bgee_top_score' in t}).T

Rn: 18134 of 23141 genes tested by Bgee, 17802 expressed somewhere, 22 tissues
Fc: 18862 of 19588 genes tested by Bgee, 18596 expressed somewhere, 12 tissues


,bgee_top_score,bgee_tissues_present
Rn,18134,18134
Fc,18862,18862


## 3 · The human ortholog of each gene

What lets annotation transfer from human the way the mouse space does. One-to-one only: a gene with several human counterparts has no single one to copy from, and picking one would invent a certainty the homology does not have. The count of the other kinds is kept so the loss is visible.

In [4]:
ORTHO = {RAT_SPACE: 'rn_human_orthologs.tsv', CAT_SPACE: 'fc_human_orthologs.tsv'}
for space, table in tables.items():
    if space not in ORTHO:
        continue                       # a human ortholog means nothing for a mosquito or a parasite
    path = os.path.join(OUT, ORTHO[space])
    if not os.path.exists(path):
        print(space.code, 'no ortholog file'); continue
    links = pd.read_csv(path, sep='	')
    one = links[links.homology_type == 'ortholog_one2one'].drop_duplicates('gene_id')
    table['human_ortholog'] = table.gene_id.map(one.set_index('gene_id').human_gene_id)
    table['human_ortholog_high_confidence'] = table.gene_id.map(
        one.set_index('gene_id').is_high_confidence).astype('Float64')
    other = links[links.homology_type != 'ortholog_one2one'].gene_id.nunique()
    print(f'{space.code}: {int(table.human_ortholog.notna().sum())} genes with a one-to-one human '
          f'ortholog; {other} more have only one-to-many or many-to-many and are left empty')
pd.DataFrame({s.code: t[['human_ortholog', 'human_ortholog_high_confidence']].notna().sum()
              for s, t in tables.items() if 'human_ortholog' in t}).T

Rn: 15541 genes with a one-to-one human ortholog; 3191 more have only one-to-many or many-to-many and are left empty
Fc: 16053 genes with a one-to-one human ortholog; 1281 more have only one-to-many or many-to-many and are left empty


,human_ortholog,human_ortholog_high_confidence
Rn,15541,15541
Fc,16053,16053


## 4 · What the parasite space already has on disk

The P. vivax arm of GSE220039 -- the same humanised-liver series read for falciparum, whose vivax half had never been opened. **Controls only**: the series pairs them with a PI4K-inhibitor arm, which is a drug experiment and a different question. Counts are renormalised to the PARASITE total, because the parasite is a small minority of a liver biopsy.

In [5]:
import numpy as np
PV_LIVER = os.path.join('transcription', 'RNAseq', 'GSE220039',
                        'GSE220039_PvHsMmu_GRCh38_GRCm39_P01v58_raw_counts.csv.gz')
pv_path = os.path.join(root, PV_LIVER)
if VIVAX_SPACE in tables and os.path.exists(pv_path):
    counts = pd.read_csv(pv_path)
    counts = counts.rename(columns={counts.columns[0]: 'gene_id'})
    counts['gene_id'] = counts.gene_id.astype(str).str.strip()
    parasite = counts[counts.gene_id.str.startswith('PVP01_')].set_index('gene_id')
    controls = [c for c in parasite.columns if 'Cont' in c]
    total = parasite[controls].to_numpy().sum()
    cpm = 1e6 * parasite[controls].sum(axis=1) / total
    table = tables[VIVAX_SPACE]
    table['expr_pv_liver'] = table.gene_id.map(np.log2(cpm + 1).round(4))
    hashes['Pv_liver'] = sha(pv_path)
    print(f'Pv: {len(controls)} control samples, {int(total):,} parasite reads, '
          f'{int(table.expr_pv_liver.notna().sum())} of {len(table)} genes with a value')
    print('   drug arm excluded:', [c for c in parasite.columns if 'PI4K' in c])
else:
    print('no vivax liver counts on disk')
tables[VIVAX_SPACE].head(3) if VIVAX_SPACE in tables else 'no vivax table'

Pv: 3 control samples, 9,054,043 parasite reads, 6523 of 6523 genes with a value
   drug arm excluded: ['p_viv_PI4K-181', 'p_viv_PI4K-182', 'p_viv_PI4K-183']


,gene_id,gene_name,biotype,seqid,start,end,strand,expr_pv_liver
0,PVP01_0100100,,protein_coding,PvP01_01_v2,2235,3011,-,0
1,PVP01_0100200,,protein_coding,PvP01_01_v2,4988,6673,+,0
2,PVP01_0100300,,protein_coding,PvP01_01_v2,9820,11476,+,0


## 5 · What the vector space already has on disk

The Anopheles arm of GSE284537 -- the same infected-midgut series read for the Plasmodium mosquito-stage slot, whose mosquito half had never been opened. The **control (dsGFP)** arms only, pseudo-bulked per timepoint: the series pairs each with an ecdysone-receptor knockdown, which is a perturbation and a different question. These are the mosquito's own genes responding to infection.

In [6]:
import glob, gzip, re
AG_DIR = os.path.join('transcription', 'scRNAseq', 'GSE284537')
ag_dir = os.path.join(root, AG_DIR)
matrices = sorted(glob.glob(os.path.join(ag_dir, '*_AnG_*matrix.mtx.gz')))
if ANOPHELES_SPACE in tables and matrices:
    per_timepoint = {}
    for matrix in matrices:
        tag = re.search(r'_AnG_r\d-(\w+)-GFP', os.path.basename(matrix))
        features = matrix.replace('matrix.mtx.gz', 'features.tsv.gz')
        if tag is None or not os.path.exists(features):
            continue
        with gzip.open(features, 'rt') as handle:
            genes = [line.split('\t')[0] for line in handle.read().splitlines()]
        totals = np.zeros(len(genes))
        with gzip.open(matrix, 'rt') as handle:
            line = handle.readline()
            while line.startswith('%'):
                line = handle.readline()
            for line in handle:
                gene, _cell, value = line.split()
                totals[int(gene) - 1] += int(value)
        per_timepoint.setdefault(tag.group(1), []).append(pd.Series(totals, index=genes))
    table = tables[ANOPHELES_SPACE]
    for timepoint in ('36h', '48h', '4d', '7d'):
        replicates = per_timepoint.get(timepoint)
        if not replicates:
            continue
        summed = sum(replicates)
        if summed.sum() <= 0:
            continue
        table[f'expr_midgut_{timepoint}'] = table.gene_id.map(
            np.log2(1e6 * summed / summed.sum() + 1).round(4))
    made = [c for c in table.columns if c.startswith('expr_midgut_')]
    print(f'Ag: {len(made)} timepoints, {int(table[made].notna().all(axis=1).sum())} of {len(table)}'
          f' genes with a value in each')
else:
    print('no Anopheles matrices on disk')
tables[ANOPHELES_SPACE].head(3) if ANOPHELES_SPACE in tables else 'no vector table'

Ag: 4 timepoints, 12574 of 12614 genes with a value in each


,gene_id,gene_name,biotype,seqid,start,end,strand,expr_midgut_36h,expr_midgut_48h,expr_midgut_4d,expr_midgut_7d
0,AGAP029842,,protein_coding,2L,17351,18787,+,0.0073,0.4684,0.0548,0.5502
1,AGAP004677,,protein_coding,2L,157348,186936,-,4.049,3.651,5.648,4.223
2,AGAP004678,,protein_coding,2L,203779,205293,+,2.683,1.763,1.515,1.739


## 6 · Validate and write

`spaces.validate_nodes` is the same check every space passes: unique, non-missing, canonical ids. A space whose ids do not match its own pattern is refused here rather than at first use.

In [7]:
from parascape.spaces import validate_nodes
written = {}
for space, table in tables.items():
    validate_nodes(space, table)
    where = os.path.join(OUT, space.nodes)
    table.to_parquet(where, index=False)
    written[space.code] = dict(rows=len(table), columns=list(table.columns), file=space.nodes,
                               build=SOURCES[space], build_sha256=hashes.get(space.code, ''))
json.dump(written, open(os.path.join(OUT, 'manifest.json'), 'w'), indent=1)
print('written:', {k: v['rows'] for k, v in written.items()})
pd.DataFrame(written).T[['rows', 'file', 'build']]

written: {'Rn': 23141, 'Fc': 19588, 'Ag': 12614, 'As': 13336, 'Pv': 6523}


,rows,file,build
Rn,23141,rn_gene_nodes.parquet,"(reference/ensembl_r113, Rattus_norvegicus.mRatBN7.2.113.gff3.gz)"
Fc,19588,fc_gene_nodes.parquet,"(reference/ensembl_r113, Felis_catus.Felis_catus_9.0.113.gff3.gz)"
Ag,12614,ag_gene_nodes.parquet,"(reference/ensembl_metazoa_r60, Anopheles_gambiae.AgamP4.60.chr.gff3.gz)"
As,13336,as_gene_nodes.parquet,"(reference/ensembl_metazoa_r60, Anopheles_stephensi.AsteS1.60.gff3.gz)"
Pv,6523,pv_gene_nodes.parquet,"(reference/ensembl_protists_r63, Plasmodium_vivax_gca900093555.GCA900093555v2.63.gff3.gz)"


## 7 · What these spaces cannot do yet

Neither holds a measurement, so neither can answer a question or be calibrated. What each needs first is stated here so the gap is a list rather than a feeling.

In [8]:
needs = pd.DataFrame([
    dict(space='Rn', first_need='an infection series: no rat Toxoplasma transcriptome is on disk',
         then='a pack, so the space can be installed and leave results/'),
    dict(space='Fc', first_need='the enteroepithelial series -- the stages that exist in no other host',
         then='a pack, so the space can be installed and leave results/'),
    dict(space='Ag', first_need='the salivary gland, where the sporozoite waits; the midgut is now covered',
         then='a pack, so the space can be installed and leave results/'),
    dict(space='As', first_need='any stephensi measurement at all: none is on disk',
         then='the urban-breeding and insecticide-resistance data this species is studied for'),
    dict(space='Pv', first_need='hypnozoite data -- the dormant stage falciparum does not have',
         then='a pack, so the space can be installed and leave results/'),
])
needs.to_csv(os.path.join(OUT, 'what_each_space_needs.tsv'), sep='\t', index=False)
needs

,space,first_need,then
0,Rn,an infection series: no rat Toxoplasma transcriptome is on disk,"a pack, so the space can be installed and leave results/"
1,Fc,the enteroepithelial series -- the stages that exist in no other host,"a pack, so the space can be installed and leave results/"
2,Ag,"the salivary gland, where the sporozoite waits; the midgut is now covered","a pack, so the space can be installed and leave results/"
3,As,any stephensi measurement at all: none is on disk,the urban-breeding and insecticide-resistance data this species is studied for
4,Pv,hypnozoite data -- the dormant stage falciparum does not have,"a pack, so the space can be installed and leave results/"
